In [9]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster

In [ ]:
# 1. Loading data for geospatial mapping

df = pd.read_csv('airbnb_ready_for_ml.csv')

# Create the map's center point (average latitude/longitude of the listings)
map_center = [df['latitude'].mean(), df['longitude'].mean()]

# Create the base map (clean and modern style)
base_map = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

In [11]:
# LAYER 1: HEATMAP (Listing Density)

# 2. Generating Heatmap (Property Density)

# The heatmap shows where there is a higher concentration of Airbnbs
heat_data = df[['latitude', 'longitude']].values.tolist()
HeatMap(heat_data, radius=15, blur=10, name="Listing Density").add_to(base_map)

In [13]:
# LAYER 2: PRICE CLUSTERS (Where is the money?)

# 3. Grouping price markers

# Cluster so the map doesn't get "cluttered" with thousands of points at once
marker_cluster = MarkerCluster(name="Individual Prices").add_to(base_map)

# Added the top 500 listings with the best rating so the map doesn't get too heavy
df_top = df.sort_values(by='review_scores_rating', ascending=False).head(500)

for idx, row in df_top.iterrows():
    # Create a text popup for when details are needed
    # (HTML <br> tags used for proper line breaks)
    popup_text = f"""
    <div style='width: 200px; font-family: Arial;'>
        <b>Neighborhood:</b> {row['neighbourhood_cleansed']}<br>
        <b>Price:</b> {row['price']}€/night<br>
        <b>Rating:</b> {row['review_scores_rating']}⭐<br>
        <b>Success:</b> {row['reviews_per_month']} rev/month<br>
        <a href='https://www.airbnb.com/rooms/{int(row['id'])}' target='_blank'>View on Airbnb</a>
    </div>
    """
    
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_text, max_width=300),
        icon=folium.Icon(color='red', icon='home')
    ).add_to(marker_cluster)

# Add layer control (allows the user to toggle the heatmap or markers on/off)
folium.LayerControl().add_to(base_map)

# Save the map as an HTML file
base_map.save('interactive_airbnb_map.html')